In [7]:
import pandas as pd
import numpy as np
from scipy import stats
import statsmodels.api as sm
from statsmodels.stats.outliers_influence import variance_inflation_factor

df = pd.read_csv('plrx.txt', delimiter='\t', header=None)
X = df.iloc[:, :12]
y = df.iloc[:, 12]

feature_names = [f'c{i+1}' for i in range(12)]
X.columns = feature_names

class_1 = X[y == 1.0]  # Relaxed
class_2 = X[y == 2.0]  # Planning

In [ ]:
import numpy as np
import pandas as pd
import pywt
from statsmodels.regression.linear_model import yule_walker

# ============================================================
# 1. Data Loading & Preparation
# ============================================================
df = pd.read_csv('plrx.txt', delimiter='\t', header=None)

# Split features and labels
X = df.iloc[:, :12]
y = df.iloc[:, 12]

# Rename columns
feature_names = [f'c{i+1}' for i in range(12)]
X.columns = feature_names

# ============================================================
# 2. Feature Extraction Definitions
# ============================================================
def hjorth_params(signal):
    first_deriv = np.diff(signal)
    second_deriv = np.diff(first_deriv)

    var_zero = np.var(signal)
    var_d1 = np.var(first_deriv)
    var_d2 = np.var(second_deriv)

    activity = var_zero
    mobility = np.sqrt(var_d1 / var_zero) if var_zero > 0 else 0
    complexity = (np.sqrt(var_d2 / var_d1) / mobility) if (var_d1 > 0 and mobility > 0) else 0

    return activity, mobility, complexity

def sample_entropy(signal, m=2, r=None):
    """
    m: embedding dimension
    r: tolerance (typically 0.2 * std of signal)
    """
    signal = np.asarray(signal)
    N = len(signal)
    if r is None:
        r = 0.2 * np.std(signal)

    def _phi(m):
        templates = np.array([signal[i:i + m] for i in range(N - m + 1)])
        count = 0
        total = 0
        for i in range(len(templates)):
            dist = np.max(np.abs(templates - templates[i]), axis=1)
            count += np.sum(dist <= r) - 1  # exclude self-match
            total += len(templates) - 1
        return count / total if total > 0 else 0

    phi_m = _phi(m)
    phi_m1 = _phi(m + 1)

    if phi_m == 0 or phi_m1 == 0:
        return 0  # avoid log(0); handle degenerate case
    return -np.log(phi_m1 / phi_m)

def ar_coefficients(signal, order=3):
    """
    Fits an AR model of given order and returns the coefficients.
    Adjusted order to 3 to accommodate 12-sample signal length.
    """
    signal = np.asarray(signal) - np.mean(signal)  # remove DC offset
    try:
        rho, sigma = yule_walker(signal, order=order, method='mle')
        return rho  # array of length `order`
    except Exception:
        return np.zeros(order) # Fallback if matrix is singular

def wpt_band_energy(signal, wavelet='db2', maxlevel=2):
    """
    Decomposes signal into full wavelet packet tree.
    Adjusted to db2 and maxlevel 2 to accommodate short 12-sample signals.
    """
    wp = pywt.WaveletPacket(data=signal, wavelet=wavelet, mode='symmetric', maxlevel=maxlevel)
    nodes = [node.path for node in wp.get_level(maxlevel, order='freq')]

    energies = {}
    total_energy = 0
    for node_path in nodes:
        coeffs = wp[node_path].data
        e = np.sum(coeffs ** 2)
        energies[f'wpt_energy_{node_path}'] = e
        total_energy += e

    for node_path in nodes:
        energies[f'wpt_relenergy_{node_path}'] = (
            energies[f'wpt_energy_{node_path}'] / total_energy if total_energy > 0 else 0
        )

    return energies

def extract_all_features(signal, ar_order=3, wpt_wavelet='db2', wpt_level=2):
    features = {}

    # Hjorth
    activity, mobility, complexity = hjorth_params(signal)
    features['hjorth_activity'] = activity
    features['hjorth_mobility'] = mobility
    features['hjorth_complexity'] = complexity

    # Sample entropy
    features['sample_entropy'] = sample_entropy(signal)

    # AR coefficients
    ar_coefs = ar_coefficients(signal, order=ar_order)
    for i, c in enumerate(ar_coefs):
        features[f'ar_coef_{i+1}'] = c

    # Multi-band WPT energy
    wpt_energies = wpt_band_energy(signal, wavelet=wpt_wavelet, maxlevel=wpt_level)
    features.update(wpt_energies)

    return features

print("Extracting features... This may take a moment depending on dataset size.")
feature_rows = [extract_all_features(row) for row in X.values]

X_all_features = pd.DataFrame(feature_rows)

print(f"\nOriginal shape: {X.shape}")
print(f"New Extracted Features shape: {X_all_features.shape}")
print("\nFirst 5 rows of extracted features:")
print(X_all_features.head())

Extracting features... This may take a moment depending on dataset size.

Original shape: (182, 12)
New Extracted Features shape: (182, 15)

First 5 rows of extracted features:
   hjorth_activity  hjorth_mobility  hjorth_complexity  sample_entropy  \
0         0.042331         1.170131           1.275784             0.0   
1         0.059279         1.743326           1.079981             0.0   
2         0.162019         1.254355           1.314821             0.0   
3         0.235432         1.191728           1.232746             0.0   
4         0.052572         1.331197           1.134159             0.0   

   ar_coef_1  ar_coef_2  ar_coef_3  wpt_energy_aa  wpt_energy_ad  \
0   0.317008  -0.137387  -0.212768       0.638633       0.112620   
1  -0.624509  -0.448333  -0.109160       0.148208       0.113396   
2   0.202767   0.148310  -0.513534       1.587944       0.547823   
3   0.373430  -0.465278  -0.076742       3.377948       1.475891   
4   0.004216  -0.246982  -0.082265    

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.preprocessing import StandardScaler

# 1. Standardize the newly extracted features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_all_features)

# 2. Fit LDA (reduces to 1 dimension for 2 classes)
lda = LinearDiscriminantAnalysis(n_components=1)
X_lda = lda.fit_transform(X_scaled, y)

# 3. Prepare data for plotting
lda_df = pd.DataFrame(X_lda, columns=["LD1"])
lda_df["Class"] = y.values

# 4. Plot overlapping histograms
plt.figure(figsize=(8, 5))
plt.hist(
    lda_df[lda_df["Class"] == 1.0]["LD1"],
    alpha=0.6,
    label="Relaxed (1.0)",
    bins=20,
    color="steelblue",
    edgecolor="black"
)
plt.hist(
    lda_df[lda_df["Class"] == 2.0]["LD1"],
    alpha=0.6,
    label="Planning (2.0)",
    bins=20,
    color="darkorange",
    edgecolor="black"
)

plt.xlabel("Linear Discriminant 1 (LD1)")
plt.ylabel("Frequency")
plt.legend()
plt.title("LDA Histogram: Separability of Extracted Features")
plt.show()

# Print the discrimination power
print(f"Explained Variance Ratio: {lda.explained_variance_ratio_[0]:.2%}")